# 02 — Explore the comprehensive comparison

Pulls every results JSON in `outputs/logs/` and renders the headline numbers,
per-network heatmap, FC-translation correlations, subject-level CV gap, and
null z-scores — all inline.

Re-run after `pipeline/05_evaluate.py` (and friends) to get a fresh view.

In [1]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'src'))
warnings.filterwarnings('ignore', category=DeprecationWarning, module='homer\\..*')

import json
import numpy as np
import pandas as pd
from homer.viz.reports import build_comparison_table, render_summary_md
from homer.viz.notebook import plot_per_network_heatmap, plot_comparison_bars

ROOT = Path.cwd().parent
LOG  = ROOT / 'outputs' / 'logs'
CMP  = ROOT / 'outputs' / 'comparison'

## 1. Build the comparison table

One function call: pulls every JSON in `outputs/logs/` and returns a wide DataFrame, a long DataFrame, the null z-score dict, and the bootstrap dict.

In [2]:
wide_df, long_df, null_z, bootstrap = build_comparison_table(LOG)
print(f'{len(wide_df)} configs × {wide_df.shape[1]} columns in wide table')
print(f'{len(long_df)} (config, network) cells in long table')
wide_df.head()

16 configs × 24 columns in wide table
156 (config, network) cells in long table


,config,label,notes,anchor_top1,anchor_top5,anchor_pair,anchor_hemi,anchor_mean_rank,anchor_mean_xyz_dist,n_anchors_total,...,full_in_neighborhood,full_mass_on_correct,fc_translation_r,fc_translation_within_net,fc_translation_cross_net,fc_translation_n_kept,subject_cv_train_r,subject_cv_test_r,subject_cv_test_r_std,subject_cv_gap
0,baseline_fc_only,baseline (FC only),,0.785714,1.0,0.785714,1.000000,1.261905,0.021166,42,...,NaN,NaN,0.364281,0.447025,0.199168,1443,0.359697,0.318918,0.005514,-0.040779
1,fc_plus_xyz_gw,FC + xyz GW,,0.809524,1.0,0.809524,1.000000,1.238095,0.019977,42,...,NaN,NaN,0.365184,0.448727,0.202508,1499,NaN,NaN,NaN,NaN
2,fc_plus_network_mask,FC + network mask,,0.809524,1.0,0.809524,1.000000,1.238095,0.019977,42,...,NaN,NaN,0.379543,0.492637,0.169893,835,NaN,NaN,NaN,NaN
3,fc_plus_SC,FC + SC (production),production,0.809524,1.0,0.809524,1.000000,1.238095,0.019977,42,...,0.071429,0.023917,0.361017,0.443506,0.198248,1379,0.356657,0.317814,0.006123,-0.038843
4,fc_plus_gene_GW,FC + gene GW,,0.761905,NaN,0.809524,0.952381,NaN,NaN,42,...,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN


## 2. Headline metrics across all configs

In [3]:
headline_cols = [
    'label', 'anchor_top1', 'anchor_top5', 'anchor_pair', 'anchor_hemi',
    'anchor_mean_rank', 'anchor_mean_xyz_dist',
    'fc_translation_r', 'subject_cv_test_r', 'notes',
]
view = wide_df[headline_cols].copy()
# Format percentages for readability
for c in ('anchor_top1', 'anchor_top5', 'anchor_pair', 'anchor_hemi'):
    view[c] = view[c].apply(lambda x: f'{x:.0%}' if np.isfinite(x) else '—')
view

,label,anchor_top1,anchor_top5,anchor_pair,anchor_hemi,anchor_mean_rank,anchor_mean_xyz_dist,fc_translation_r,subject_cv_test_r,notes
0,baseline (FC only),79%,100%,79%,100%,1.261905,0.021166,0.364281,0.318918,
1,FC + xyz GW,81%,100%,81%,100%,1.238095,0.019977,0.365184,NaN,
2,FC + network mask,81%,100%,81%,100%,1.238095,0.019977,0.379543,NaN,
3,FC + SC (production),81%,100%,81%,100%,1.238095,0.019977,0.361017,0.317814,production
4,FC + gene GW,76%,—,81%,95%,NaN,NaN,NaN,NaN,
5,FC + M_gene,60%,—,64%,93%,NaN,NaN,NaN,NaN,
6,FC + SC + M_gene,62%,—,69%,90%,NaN,NaN,NaN,NaN,
7,all modalities (FC+xyz+SC+gene),64%,—,71%,90%,NaN,NaN,NaN,NaN,
8,FC + selective M_gene,60%,—,64%,93%,NaN,NaN,NaN,NaN,
9,FC + SC + selective M_gene,62%,—,69%,90%,NaN,NaN,NaN,NaN,


## 3. Multi-metric bar chart

Production config (`fc_plus_SC`) shown in orange. The 4 panels are anchor top-1, anchor top-5, mean xyz distance (lower = better), and FC translation r.

## 2b. Full-space recovery — the honest per-voxel metric

The headline `anchor_top1` above is *restricted* to held-out anchor candidates. The
table below uses the global argmax over all 2094 human nodes. **Big drop expected** —
the model's full-space argmax usually lands on a non-anchor grid node *near* the correct
anchor, not the anchor itself.

In [4]:
fs_cols = ['label', 'full_top1', 'full_top5', 'full_mean_rank',
           'full_argmax_is_anchor', 'full_in_neighborhood', 'full_mass_on_correct',
           'notes']
fs_view = wide_df.dropna(subset=['full_top1'])[fs_cols].copy()
for c in ('full_top1', 'full_top5', 'full_argmax_is_anchor', 'full_in_neighborhood'):
    fs_view[c] = fs_view[c].apply(lambda x: f'{x:.0%}' if np.isfinite(x) else '—')
for c in ('full_mean_rank',):
    fs_view[c] = fs_view[c].apply(lambda x: f'{x:.0f}' if np.isfinite(x) else '—')
for c in ('full_mass_on_correct',):
    fs_view[c] = fs_view[c].apply(lambda x: f'{x:.3f}' if np.isfinite(x) else '—')
fs_view

,label,full_top1,full_top5,full_mean_rank,full_argmax_is_anchor,full_in_neighborhood,full_mass_on_correct,notes
3,FC + SC (production),2%,12%,206,5%,7%,0.024,production


In [5]:
plot_comparison_bars(wide_df)

## 4. Per-network heatmap

The variance across networks is the story — most configs land 100% on the
easy networks (auditory, frontoparietal, frontal_dmn, ...) and 25-50% on
the hard ones (visual, brainstem, sensorimotor, salience, subcortical).

In [6]:
plot_per_network_heatmap(long_df)

## 5. Null calibration

How does the production model do vs random π and permuted-anchor null distributions?

In [7]:
null_table = pd.DataFrame([
    {
        'null_kind':   k,
        'n_trials':    v['n_trials'],
        'real_top1':   f'{v["real_top1"]:.0%}',
        'null_mean':   f'{v["null_mean"]:.0%}',
        'null_std':    f'{v["null_std"]:.0%}',
        'z_score':     f'{v["z_score"]:+.1f}',
    }
    for k, v in null_z.items()
])
null_table

,null_kind,n_trials,real_top1,null_mean,null_std,z_score
0,random_pi,50,81%,28%,7%,+7.5
1,permuted_anchors,5,81%,31%,3%,+17.8


## 6. Subject-level cross-validation (item D)

For the two configs we ran subject CV on, what's the train/test gap?

In [8]:
subj_path = LOG / 'subject_cv.json'
if subj_path.exists():
    sv = json.loads(subj_path.read_text())
    rows = []
    for cfg_key, folds in sv.items():
        fr = list(folds.values())
        rows.append({
            'config': cfg_key.split('__')[0],
            'n_folds':    len(fr),
            'train_r':    f'{np.mean([f["train_r_overall"] for f in fr]):.3f} ± '
                          f'{np.std([f["train_r_overall"] for f in fr]):.3f}',
            'test_r':     f'{np.mean([f["test_r_overall"] for f in fr]):.3f} ± '
                          f'{np.std([f["test_r_overall"] for f in fr]):.3f}',
            'gap':        f'{np.mean([f["test_minus_train"] for f in fr]):+.3f} ± '
                          f'{np.std([f["test_minus_train"] for f in fr]):.3f}',
        })
    pd.DataFrame(rows)
else:
    print('No subject_cv.json yet — run experiments/D_subject_cv/subject_cv.py first.')
    pd.DataFrame()

## 7. Bootstrap stability

From 40 subject-level bootstrap iterations of the production model.

In [9]:
# Note: post-fix bootstrap files are per-config (fc_plus_SC, fc_only).
# build_comparison_table prefers the SC one. Show both shapes for completeness.
if not bootstrap:
    print('No bootstrap_summary file yet — run pipeline/06_bootstrap.py.')
else:
    keys_old = ['n_iterations', 'mean_stability', 'median_stability',
                'frac_stable_above_0.8', 'frac_stable_above_0.5']
    keys_new = ['config', 'n_iterations',
                'argmax_row_stability_mean', 'argmax_row_stability_median',
                'argmax_row_frac_above_0.8', 'argmax_row_frac_above_0.5',
                'argmax_row_frac_perfect',
                'cell_stability_mean', 'cell_frac_stable_above_0.8']
    keys = keys_new if 'argmax_row_stability_mean' in bootstrap else keys_old
    for k in keys:
        v = bootstrap.get(k)
        if v is None: continue
        if isinstance(v, float) and 0 <= v <= 1 and ('frac' in k or 'stability' in k):
            print(f'  {k:35s} {v:.1%}')
        elif isinstance(v, float):
            print(f'  {k:35s} {v:.4f}')
        else:
            print(f'  {k:35s} {v}')

  config                              fc_plus_SC
  n_iterations                        40
  argmax_row_stability_mean           97.8%
  argmax_row_stability_median         100.0%
  argmax_row_frac_above_0.8           95.0%
  argmax_row_frac_above_0.5           99.4%
  argmax_row_frac_perfect             88.4%
  cell_stability_mean                 100.0%
  cell_frac_stable_above_0.8          100.0%


## 8. The full markdown summary

Same content as `outputs/comparison/comparison_summary.md`. Copy/paste-ready for reports.

In [10]:
from IPython.display import Markdown, display
display(Markdown(render_summary_md(wide_df, long_df, null_z, bootstrap)))

# Comprehensive comparison — all configs, all metrics

Generated from results in `outputs/logs/` on data of 40-iter bootstrap and 11-network leave-one-network-out CV. Production config marked **bold**.

## Headline table — restricted-anchor CV (the 'ranking' metric)

Top-1 here is **argmax restricted to the held-out anchor candidates**, NOT global argmax over all 2094 human nodes (which is in the next table).

FC-translation r is **in-sample** (uses the same fc_mean to build C_h that it evaluates).

| Config | rTop-1 | rTop-5 | Pair | Hemi | Rank | xyz_d | FC-r overall | FC-r within | FC-r cross | Subj-CV test r | Notes |
|---|---|---|---|---|---|---|---|---|---|---|---|
| baseline (FC only) | 79% | 100% | 79% | 100% | 1.26 | 0.021 | 0.36 | 0.45 | 0.20 | 0.32 |  |
| FC + xyz GW | 81% | 100% | 81% | 100% | 1.24 | 0.020 | 0.37 | 0.45 | 0.20 | — |  |
| FC + network mask | 81% | 100% | 81% | 100% | 1.24 | 0.020 | 0.38 | 0.49 | 0.17 | — |  |
| **FC + SC (production)** | 81% | 100% | 81% | 100% | 1.24 | 0.020 | 0.36 | 0.44 | 0.20 | 0.32 | production |
| FC + gene GW | 76% | — | 81% | 95% | — | — | — | — | — | — |  |
| FC + M_gene | 60% | — | 64% | 93% | — | — | — | — | — | — |  |
| FC + SC + M_gene | 62% | — | 69% | 90% | — | — | — | — | — | — |  |
| all modalities (FC+xyz+SC+gene) | 64% | — | 71% | 90% | — | — | — | — | — | — |  |
| FC + selective M_gene | 60% | — | 64% | 93% | — | — | — | — | — | — |  |
| FC + SC + selective M_gene | 62% | — | 69% | 90% | — | — | — | — | — | — |  |
| FC + M_anchor (item A) | 69% | 100% | 69% | 95% | 1.60 | 0.031 | — | — | — | — | negative |
| FC + SC + M_anchor (item A) | 69% | 100% | 69% | 95% | 1.60 | 0.034 | — | — | — | — | negative |
| Hierarchical (per-network) | 45% | 93% | 67% | 64% | 2.36 | 0.160 | 0.39 | 0.55 | 0.16 | — | M4: cleaner WN, hurts CV |
| Iterative soft (lam=0.30, item B) | 50% | 100% | 50% | 100% | 1.75 | 0.053 | — | — | — | — | no-op; 2/11 nets only |
| Iterative hard (lam=1.00, item B) | 81% | 100% | 81% | 100% | 1.24 | 0.020 | — | — | — | — | no-op |

## Full-space recovery — global argmax over all 2094 human nodes

The HONEST per-voxel metric. The model's full-space argmax typically lands on a non-anchor *grid* node near the correct anchor rather than the anchor itself. Full-top-1 is much smaller than restricted-top-1 because the search space is 2094× larger.

| Config | full-Top-1 | full-Top-5 | mean rank /2094 | argmax is anchor | in 5% neighborhood | mean mass on correct anchor |
|---|---|---|---|---|---|---|
| **FC + SC (production)** | 2% | 12% | 206 | 5% | 7% | 0.024 |

## Null calibration (production = `fc_plus_SC`)

Each cell of the null is a per-trial weighted-mean top-1 across all 11 networks.

| Null kind | n trials | Real top-1 | Null mean | Null std | z-score |
|---|---|---|---|---|---|
| random_pi | 50 | 81% | 28% | 7% | +7.5 |
| permuted_anchors | 5 | 81% | 31% | 3% | +17.8 |

## Bootstrap stability (production solve)

- iterations: 40
- mean per-cell stability: 0.000
- median: 0.000
- frac stable above 0.8: 0.0%
- frac stable above 0.5: 0.0%

## Per-network top-1 heatmap (see fig 14)

Variance across networks is the story — most configs land 100% on the easy networks (auditory, frontoparietal, frontal_dmn, etc.) and 25-50% on visual / brainstem / sensorimotor.


## 9. Save current snapshot to `outputs/comparison/`

(Same files `pipeline/07_build_artefacts.py` produces — useful if you've changed things in this notebook and want to persist.)

In [11]:
from homer.viz.reports import make_comparison_bars_figure, make_per_network_heatmap_figure
CMP.mkdir(parents=True, exist_ok=True)
wide_df.to_csv(CMP / 'comprehensive_table.csv', index=False, float_format='%.4f')
long_df.to_csv(CMP / 'per_network_top1.csv', index=False, float_format='%.4f')
(CMP / 'comparison_summary.md').write_text(render_summary_md(wide_df, long_df, null_z, bootstrap))
print(f'saved → {CMP}/comprehensive_table.csv')
print(f'saved → {CMP}/per_network_top1.csv')
print(f'saved → {CMP}/comparison_summary.md')

saved → /Users/Peach_R/Dropbox/Work/ResearchProjects/brain_crossspecies_translation/homer/outputs/comparison/comprehensive_table.csv
saved → /Users/Peach_R/Dropbox/Work/ResearchProjects/brain_crossspecies_translation/homer/outputs/comparison/per_network_top1.csv
saved → /Users/Peach_R/Dropbox/Work/ResearchProjects/brain_crossspecies_translation/homer/outputs/comparison/comparison_summary.md


## 10. Trust map — where to (and not to) trust the model

Per-parcel trust signal computed in `pipeline/05g_compute_trust.py`. We
expose two views:

- **Regional empirical trust** — per-region top-1 against Beauchamp 2022.
  This is the honest signal: emerald = top-1 ≥15% in this region (Thalamus,
  Auditory, Somatosensory), amber = 3-15%, red = <3%, grey = parcel not in
  any of our 19 evaluable Beauchamp regions.
- **Model-confidence trust** — composite of bootstrap stability +
  argmax mass concentration + FC similarity to nearest anchor. Less
  informative (88% of parcels have perfect bootstrap & concentration,
  so it mostly reflects FC similarity).

The regional view is what you should use to decide whether to trust a
specific prediction.


In [12]:
import numpy as np
from homer.data import load_cached
from homer.viz.notebook import plot_brain_3d

M, _ = load_cached('mouse', cache_dir='../outputs/anndata')
ts = np.load('../outputs/coupling/trust_score_fc_plus_SC.npz')

# Show the regional-tier 3D map
fig = plot_brain_3d(
    M,
    color_by='trust_tier',
    trust_tier=ts['regional_tier'],
    title='Mouse parcels — regional trust tier (production π, fc_plus_SC)',
)
fig.show()


**Reading the colors.** Emerald regions are reliable (Thalamus,
Auditory cortex, Somatosensory cortex — all >15% Beauchamp top-1). Amber
regions are usable as priors with care (Caudoputamen, Cingulate,
Hypothalamus, Visual, etc.). Red regions are where the model fails
(midbrain colliculi, hippocampus without supplementary anchors, motor with
broad anchor only). Grey parcels are not in any validated Beauchamp region
— absence of evidence, not evidence of absence.

In [13]:
# Compare to the augmented π (with M1 + hippocampal supplementary anchors)
import os
aug_path = '../outputs/coupling/trust_score_fc_plus_SC_with_M1_hippo.npz'
if os.path.exists(aug_path):
    ts_aug = np.load(aug_path)
    fig_aug = plot_brain_3d(
        M,
        color_by='trust_tier',
        trust_tier=ts_aug['regional_tier'],
        title='Mouse parcels — regional trust tier (AUGMENTED: + M1 + hippocampal anchors)',
    )
    fig_aug.show()

    # Diff — how many parcels moved from low → medium?
    before = ts['regional_tier']; after = ts_aug['regional_tier']
    upgrades = ((before == 'low') & (after == 'medium')).sum()
    downgrades = ((before == 'medium') & (after == 'low')).sum()
    print(f'Tier upgrades (low → medium): {upgrades} parcels')
    print(f'Tier downgrades (medium → low): {downgrades} parcels')
else:
    print(f'No augmented trust file at {aug_path} — run pipeline/05g_compute_trust.py --pi-file pi_fc_plus_SC_with_M1_hippo.npy')


Tier upgrades (low → medium): 130 parcels
Tier downgrades (medium → low): 0 parcels


## 11. Per-region accuracy table

The empirical Beauchamp top-1 per region — directly readable as "how often
does the model land in the right region for parcels in this Beauchamp
group?". This drives the trust tier above.

In [14]:
import pandas as pd, json
beau = json.loads(open('../outputs/logs/beauchamp_validation.json').read())
rows = []
for pair, r in beau.items():
    if pair == '__aggregate__' or 'skip_reason' in r: continue
    rows.append({
        'pair': pair[:60],
        'n_mouse': r['n_mouse_parcels'],
        'n_human': r['n_human_parcels'],
        'top1':  r['top1'],
        'top5':  r['top5'],
        'top10': r['top10'],
        'mean_xyz_dist_mm': r['mean_xyz_dist_mm'],
        'is_anchor': r['is_anchor_overlapping'],
    })
df = pd.DataFrame(rows).sort_values('top1', ascending=False)
df.style.format({'top1': '{:.0%}', 'top5': '{:.0%}', 'top10': '{:.0%}',
                  'mean_xyz_dist_mm': '{:.1f}'})\
    .background_gradient(subset=['top1'], cmap='RdYlGn', vmin=0, vmax=0.4)


,pair,n_mouse,n_human,top1,top5,top10,mean_xyz_dist_mm,is_anchor
17,Hypothalamus -> hypothalamus,52,4,100%,100%,100%,6.8,True
18,Thalamus -> thalamus,110,22,34%,48%,55%,24.5,True
5,Anterior cingulate area -> cingulate gyrus,23,24,30%,35%,35%,34.8,True
9,Visual areas -> cuneus,54,36,17%,17%,17%,65.2,True
11,Striatum ventral region -> nucleus accumbens,26,2,15%,35%,50%,20.4,True
10,Pallidum -> globus pallidus,44,6,5%,7%,16%,20.4,True
0,Piriform area -> piriform cortex,47,13,4%,13%,26%,46.5,True
16,Pons -> pons,69,6,3%,3%,3%,52.6,True
8,Primary somatosensory area -> postcentral gyrus,155,39,2%,47%,51%,47.0,True
4,Dentate gyrus -> dentate gyrus,22,4,0%,0%,0%,53.2,False
